![Что такое распознавание именованных сущностей в обработке естественного языка](https://assets-global.website-files.com/62a9e41d28a7ab25849bce9c/62fd1b91678fbb2e45fbd7f5_What-is-Named-Entity-Recognition-in-Natural-Language-Processing.jpg)

В этом блокноте мы покажем, как выполнить распознавание именованных сущностей (NER) с помощью Hugging Face, используя модель DistilBERT.

Давайте начнём!

In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.3 MB/s eta 0:00:00


# <b><div style='padding:15px;background-color:#850E35;color:white;border-radius:2px;font-size:110%;text-align: center'>Загрузка датасета</div></b>

Датасет, который мы будем использовать, — это [wnut_17](https://huggingface.co/datasets/wnut_17), содержащий новые и редкие сущности. Начнём с его загрузки.

In [ ]:
from datasets import load_dataset
wnut = load_dataset("wnut_17")
wnut

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

wnut_17.py:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

The repository for wnut_17 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/wnut_17.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/3394 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1009 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1287 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 3394
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1009
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1287
    })
})

Как видите, датасет состоит из трёх наборов: обучающего, валидационного и тестового. Давайте посмотрим первую строку обучающего набора.

In [ ]:
print(wnut["train"][0])

{'id': '0', 'tokens': ['@paulwalk', 'It', "'s", 'the', 'view', 'from', 'where', 'I', "'m", 'living', 'for', 'two', 'weeks', '.', 'Empire', 'State', 'Building', '=', 'ESB', '.', 'Pretty', 'bad', 'storm', 'here', 'last', 'evening', '.'], 'ner_tags': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 8, 8, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0]}


Датасет включает три признака: **id**, **tokens** и **ner_tags**. Давайте рассмотрим метки NER:

In [ ]:
label_list = wnut["train"].features[f"ner_tags"].feature.names
label_list

['O',
 'B-corporation',
 'I-corporation',
 'B-creative-work',
 'I-creative-work',
 'B-group',
 'I-group',
 'B-location',
 'I-location',
 'B-person',
 'I-person',
 'B-product',
 'I-product']

Отлично, мы изучили датасет. Перейдём к предобработке.

# <b><div style='padding:15px;background-color:#850E35;color:white;border-radius:2px;font-size:110%;text-align: center'>Предобработка</div></b>

Модель, которую мы будем использовать, — это модель на основе BERT (DistilBERT). Давайте загрузим токенизатор этой модели.

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Отлично, мы загрузили токенизатор. Давайте попробуем его на тексте:

In [ ]:
example = wnut["train"][0]
tokenized_input = tokenizer(example["tokens"], is_split_into_words=True)
tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
print(tokens)

['[CLS]', '@', 'paul', '##walk', 'it', "'", 's', 'the', 'view', 'from', 'where', 'i', "'", 'm', 'living', 'for', 'two', 'weeks', '.', 'empire', 'state', 'building', '=', 'es', '##b', '.', 'pretty', 'bad', 'storm', 'here', 'last', 'evening', '.', '[SEP]']


Как видите, наш текст был токенизирован и разбит на субслова.

Чтобы токенизировать и выровнять метки, давайте создадим функцию. Эта предобработка необходима для задачи NER.

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # специальные токены
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

Отлично, давайте токенизируем весь датасет с использованием этой функции:

In [ ]:
tokenized_wnut = wnut.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/3394 [00:00<?, ? examples/s]

Map:   0%|          | 0/1009 [00:00<?, ? examples/s]

Map:   0%|          | 0/1287 [00:00<?, ? examples/s]

Перейдём к выравниванию (padding). Для этого мы будем использовать data collator.

In [ ]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

Теперь перейдём к настройке метрик. Для этого мы будем использовать библиотеки `evaluate` и `seqeval`. Сначала установим эти библиотеки.

In [ ]:
!pip install -q evaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 5.5 MB/s eta 0:00:00


Импортируем эти библиотеки.

In [ ]:
import evaluate
seqeval = evaluate.load("seqeval")

Теперь создадим функцию для вычисления метрик.

In [ ]:
import numpy as np

labels = [label_list[i] for i in example["ner_tags"]]

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

# <b><div style='padding:15px;background-color:#850E35;color:white;border-radius:2px;font-size:110%;text-align: center'>Обучение модели</div></b>

В этом разделе мы будем обучать модель на основе BERT. Сначала создадим две переменные для сопоставления меток и их идентификаторов.

In [ ]:
id2label = {
    0: "O",
    1: "B-corporation",
    2: "I-corporation",
    3: "B-creative-work",
    4: "I-creative-work",
    5: "B-group",
    6: "I-group",
    7: "B-location",
    8: "I-location",
    9: "B-person",
    10: "I-person",
    11: "B-product",
    12: "I-product"
}
label2id = {
    "O": 0,
    "B-corporation": 1,
    "I-corporation": 2,
    "B-creative-work": 3,
    "I-creative-work": 4,
    "B-group": 5,
    "I-group": 6,
    "B-location": 7,
    "I-location": 8,
    "B-person": 9,
    "I-person": 10,
    "B-product": 11,
    "I-product": 12
}

Пора загрузить модель DistilBERT:

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=13,
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Чтобы загрузить модель с Hub, давайте войдём в систему.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

Для обучения зададим гиперпараметры с помощью класса TrainingArguments.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="my_ner_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to=["none"],
    push_to_hub=True
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


После этого создадим объект Trainer.

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    train_dataset=tokenized_wnut["train"],
    eval_dataset=tokenized_wnut["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    args=training_args
)

<ipython-input-19-444553df426a>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Модель готова к дообучению. Вызовем метод train для обучения.

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.276878,0.600432,0.257646,0.360571,0.939336
2,No log,0.268231,0.603448,0.324374,0.421941,0.943141


/usr/local/lib/python3.11/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TrainOutput(global_step=426, training_loss=0.2133279540729075, metrics={'train_runtime': 62.0344, 'train_samples_per_second': 109.423, 'train_steps_per_second': 6.867, 'total_flos': 91781128898820.0, 'train_loss': 0.2133279540729075, 'epoch': 2.0})

Отлично, обучение завершено. Давайте опубликуем модель на Hub.

In [ ]:
trainer.push_to_hub()

CommitInfo(commit_url='https://huggingface.co/iroli/my_ner_model/commit/883f2366b0212cc79e02694803772b5a7b742a72', commit_message='End of training', commit_description='', oid='883f2366b0212cc79e02694803772b5a7b742a72', pr_url=None, repo_url=RepoUrl('https://huggingface.co/iroli/my_ner_model', endpoint='https://huggingface.co', repo_type='model', repo_id='iroli/my_ner_model'), pr_revision=None, pr_num=None)

# <b><div style='padding:15px;background-color:#850E35;color:white;border-radius:2px;font-size:110%;text-align: center'>Предсказание</div></b>

Для инференса возьмём текст и классифицируем его токены.

In [ ]:
from transformers import pipeline

text = "My name is Sarah, I live in London"
classifier = pipeline("ner", model=".../my_ner_model")
classifier(text)

config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/266M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cuda:0


[{'entity': 'B-person',
  'score': 0.8304478,
  'index': 4,
  'word': 'sarah',
  'start': 11,
  'end': 16},
 {'entity': 'B-location',
  'score': 0.72554874,
  'index': 9,
  'word': 'london',
  'start': 28,
  'end': 34}]

Как видите, наша модель работает хорошо. Чтобы увидеть предсказанные метки, давайте создадим функцию.

In [ ]:
import pandas as pd

def tag_sentence(text: str):
    # Преобразуем наш текст в токенизированную последовательность
    inputs = tokenizer(text, truncation=True, return_tensors="pt").to("cuda")
    # Получаем выходы модели
    outputs = model(**inputs)
    # Применяем softmax для получения вероятностей
    probs = outputs[0][0].softmax(1)
    # Получаем метки с наибольшей вероятностью для каждого токена
    word_tags = [
        (tokenizer.decode(inputs['input_ids'][0][i].item()), id2label[tagid.item()])
        for i, tagid in enumerate(probs.argmax(axis=1))
    ]

    return pd.DataFrame(word_tags, columns=['word', 'tag'])

print(tag_sentence(text))

      word         tag
0    [CLS]           O
1       my           O
2     name           O
3       is           O
4    sarah    B-person
5        ,           O
6        i           O
7     live           O
8       in           O
9   london  B-location
10   [SEP]           O


Вот и всё.

Успехов в программировании